### LLM Agents with Model Contect Protocol

This time the tools are served by a **FastMCP** server and pulled into the agent through
**langchain-mcp-adapters**. For now the agent code is almost identical to notebook 01;
only the *source* of the tools changes.

**Task:** For profile `P001`, which genes connected to autism spectrum disorder should we prioritize, and why?

```text
LLM client  ->  MCP server (FastMCP)  ->  src/mofa_tools.py  ->  structured result  ->  LLM answer
```

---

### Model backend: not decided yet — bring-your-own-key for now

> As in notebook 01, the model backend is **not finalised** (see
> `planning/session-3-framework-plan.md`). This notebook assumes a
> **bring-your-own-key** setup: it reads `ANTHROPIC_API_KEY` from a local `.env`
> file and calls the real Anthropic API via LangChain. Run it with the
> **`eccb`** kernel.

### Why MCP at all? 

> With this toy example, MCP buys us nothing over the local functions in notebook 01 — that's the point: we keep the *task* identical so the only thing you
> learn is the **interface**. We'll eventually need a different example query where the tools wrap something we *don't* want to ship inside every notebook:
>
> - **Big data** — a full knowledge graph / database that we don't want to load into the participant's Python process.
> - **Private or controlled data** — a server that holds credentials or sits
>   behind institutional access, exposing only safe, audited operations.

>
> **TODO (future):** swap the toy server for one fronting a larger/private graph
> to make this advantage concrete in the workshop.

## 0. Environment & API key

Load `ANTHROPIC_API_KEY` from `.env`. We only check that it is present — we
never print it.

In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found. Create a .env file in the project root "
        "with `ANTHROPIC_API_KEY=sk-ant-...` (bring-your-own-key)."
    )

print("ANTHROPIC_API_KEY loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

ANTHROPIC_API_KEY loaded: True


## 1. The MCP server

The server lives at `server/toy_kg_mcp_server.py`. It uses **FastMCP**
(`from mcp.server.fastmcp import FastMCP`) to expose three kinds of MCP
primitive, all backed by `src/mofa_tools.py`:

- **Tools** (model-callable, tagged read-only): `search_nodes`,
  `get_neighbors`, `rank_profile_features`, `rank_candidate_genes`
- **Resource**: `kg://summary` — a compact graph summary to read into context
- **Prompt**: `prioritize_genes(profile_id, phenotype)` — a reusable template

You can run it standalone with `python server/toy_kg_mcp_server.py`, but we
won't — in the sections below the clients launch it as a stdio subprocess for
us. (See `server/README.md` for notes on the server.)

## 2. Under the hood: talk to MCP directly (no adapter)

Before we use the convenient LangChain adapter, let's connect to the server with the MCP SDK's own client. 
This shows what MCP actually *is* before any framework adaptation.

We open a `ClientSession` over stdio, do the handshake, then enumerate **every
primitive the server exposes** — not only tools, but also resources and
prompts — and finally call a tool over the wire.

In [2]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PATH = PROJECT_ROOT / "server" / "toy_kg_mcp_server.py"
server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])

# Open a raw MCP session ourselves (no LangChain). This is the protocol as-is.
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        # 1. Handshake: client and server exchange capabilities.
        init = await session.initialize()
        print(f"Connected to: {init.serverInfo.name} v{init.serverInfo.version}")
        caps = init.capabilities
        print("Server advertises:",
              "tools" if caps.tools else "",
              "resources" if caps.resources else "",
              "prompts" if caps.prompts else "")

        # 2. TOOLS — model-callable, each with LLM-facing annotations.
        print("\n--- TOOLS (the model may choose to call these) ---")
        for t in (await session.list_tools()).tools:
            args = ", ".join(t.inputSchema.get("properties", {}))
            ro = getattr(t.annotations, "readOnlyHint", None)
            print(f"  {t.name}({args})   readOnlyHint={ro}")

        # 3. RESOURCES — context you READ, not a function you call.
        print("\n--- RESOURCES (data to read into context) ---")
        for r in (await session.list_resources()).resources:
            print(f"  {r.uri}  ({r.name})")
        summary = await session.read_resource("kg://summary")
        print("  read kg://summary ->", summary.contents[0].text[:70], "...")

        # 4. PROMPTS — reusable templates the server offers (like slash-commands).
        print("\n--- PROMPTS (server-provided templates) ---")
        for p in (await session.list_prompts()).prompts:
            args = ", ".join(a.name for a in (p.arguments or []))
            print(f"  {p.name}({args})")
        filled = await session.get_prompt(
            "prioritize_genes",
            {"profile_id": "P001", "phenotype": "autism spectrum disorder"},
        )
        print("  get_prompt ->", filled.messages[0].content.text[:70], "...")

        # 5. Actually CALL a tool over the protocol.
        print("\n--- tools/call rank_candidate_genes ---")
        called = await session.call_tool(
            "rank_candidate_genes",
            {"profile_id": "P001", "phenotype_id": "HP:0000729"},
        )
        for block in called.content:
            print("  ", block.text)


Connected to: eccb2026-toy-kg v1.28.0
Server advertises: tools resources prompts

--- TOOLS (the model may choose to call these) ---
  search_nodes(query)   readOnlyHint=True
  get_neighbors(node_id, node_type)   readOnlyHint=True
  rank_profile_features(profile_id, top_n)   readOnlyHint=True
  rank_candidate_genes(profile_id, phenotype_id)   readOnlyHint=True

--- RESOURCES (data to read into context) ---
  kg://summary  (graph_summary)
  read kg://summary -> {
  "name": "ECCB 2026 toy biomedical knowledge graph",
  "nodes": 6,
 ...

--- PROMPTS (server-provided templates) ---
  prioritize_genes(profile_id, phenotype)
  get_prompt -> For profile P001, which genes connected to autism spectrum disorder sh ...

--- tools/call rank_candidate_genes ---
   {
  "gene_id": "GENE:SHANK3",
  "name": "SHANK3",
  "profile_score": 2.4,
  "phenotype_id": "HP:0000729"
}
   {
  "gene_id": "GENE:SCN2A",
  "name": "SCN2A",
  "profile_score": 1.7,
  "phenotype_id": "HP:0000729"
}
   {
  "gene_id": "GENE

The block above never imported LangChain. We spoke to MCP **directly**, and that
exposes the shape of the protocol:

- **It's JSON-RPC 2.0 over a transport.** Here the transport is `stdio` (we
  pipe to a subprocess); it could equally be HTTP. On top of that wire, MCP
  defines named methods — `initialize`, `tools/list`, `tools/call`,
  `resources/list`, `resources/read`, `prompts/list`, `prompts/get`. That's the
  layering: **stdio/HTTP → JSON-RPC → MCP semantics**. MCP is *not* a transport
  like TCP; it rides on one.
- **Tools are one primitive of several.** A server advertises *capabilities* at
  the handshake, then offers any of:

| Primitive | Driven by | What it is | In our server |
|-----------|-----------|------------|---------------|
| **Tools** | the model | functions the model may *choose* to call | the 4 graph functions |
| **Resources** | the app/user | data to *read into context* (not "called") | `kg://summary` |
| **Prompts** | the user | reusable, parameterized prompt templates | `prioritize_genes` |
| **Sampling** | the **server** | server asks the *client's* LLM to generate | (not used) |
| **Elicitation** | the server | server asks the *human* for input mid-call | (not used) |
| **Roots** | the client | filesystem/scope boundaries the server may touch | (not used) |

- **There is LLM/human-specific metadata in the schema.** Notice each tool
  carried `readOnlyHint=True` / `destructiveHint=False`. Those *annotations*
  exist so a client can decide whether to auto-run a tool or pause for human
  approval — a concept only meaningful when a non-deterministic model is the
  caller. A generic RPC framework (gRPC, bare JSON-RPC) has no such notion.

Next we let `langchain-mcp-adapters` do this same handshake for us and convert
the **tools** into LangChain tools — while keeping the protocol (and the other
primitives, via `get_resources()` / `get_prompt()`) fully available.

## 3. Framework Adaptation: `langchain-mcp-adapters`

A clarification on scope, because it's a common misconception: we call
**`get_tools()`** here, so only the *tools* primitive flows into our agent.  
The same client also exposes `get_resources()`, `get_prompt()`. 
They just don't auto-belong in a `bind_tools()`
tool-calling loop (a resource is context to *read*; a prompt is a message
template), so we don't load them here.

**Nothing about MCP is thrown away.** The adapter *is* an MCP client — it speaks
the identical wire protocol as section 2. `get_tools()` simply converts the one
primitive a tool-calling agent consumes; the protocol, the other primitives, and
the server's reusability across clients are all intact.

> **Our choice — a more MCP-native framework?** `langchain-mcp-adapters` is
> deliberately *tool-centric*: to actually *use* resources, prompts, sampling,
> or elicitation we'd hand-wire each around a LangChain loop. If those primitives
> become central to the workshop (e.g. serving a big graph as a resource, or
> offering server-side prompt templates), it may be worth swapping for a
> framework that treats MCP as first-class and integrates it more directly.
> Candidates to evaluate:
>
> - **[Pydantic AI](https://ai.pydantic.dev/mcp/client/)** — full MCP support
>   (tools, resources, prompts, sampling, elicitation, roots), and can opt into a
>   provider's *native* MCP with one flag.
> - **[fast-agent](https://github.com/evalstate/fast-agent)** — MCP-native;
>   first framework with end-to-end sampling **and** elicitation.
> - **[lastmile mcp-agent](https://github.com/lastmile-ai/mcp-agent)** — agents
>   built directly on MCP primitives.
> - **[OpenAI Agents SDK](https://openai.github.io/openai-agents-python/mcp/)** /
>   **[Google ADK](https://google.github.io/adk-docs/)** — native MCP clients
>   (more tool-focused).
> - **Raw MCP SDK + a provider SDK** (section 2 style) — most control, most
>   boilerplate.
>
> Tracked alongside the other layer choices in
> `planning/session-3-framework-plan.md`.

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "toy_kg": {
            "command": sys.executable,
            "args": [str(SERVER_PATH)],
            "transport": "stdio",
        }
    }
)

tools = await mcp_client.get_tools()
tools_by_name = {t.name: t for t in tools}
print("Tools loaded via get_tools() (this is the tools primitive only):")
for t in tools:
    print(f"  - {t.name}: {t.description.splitlines()[0]}")


def parse_mcp(result):
    """MCP tools return a list of text content blocks; parse each block's JSON.

    FastMCP serializes one content block per list element, so we json.loads each
    block back into a Python dict. (The agent loop below doesn't need this — it
    just forwards the raw text to the model.)
    """
    import json

    if isinstance(result, list):
        parsed = []
        for block in result:
            text = block.get("text") if isinstance(block, dict) else block
            parsed.append(json.loads(text) if isinstance(text, str) else text)
        return parsed
    return result

Tools loaded via get_tools() (this is the tools primitive only):
  - search_nodes: Search graph nodes by identifier or display name.
  - get_neighbors: Return incoming and outgoing graph neighbors for a node.
  - rank_profile_features: Return the top molecular features for a toy multi-omic profile.
  - rank_candidate_genes: Rank phenotype-linked genes by their toy profile score.


In [4]:
# Proof that the adapter did NOT discard the other primitives:
# the resource and prompt are reachable through the same client.
res = await mcp_client.get_resources("toy_kg", uris=["kg://summary"])
print("get_resources('toy_kg', ['kg://summary']) ->")
print("  ", res[0].data[:80], "...\n")

pm = await mcp_client.get_prompt(
    "toy_kg", "prioritize_genes",
    arguments={"profile_id": "P001", "phenotype": "autism spectrum disorder"},
)
print("get_prompt('toy_kg', 'prioritize_genes', ...) ->")
print("  ", pm[0].content[:80], "...")

get_resources('toy_kg', ['kg://summary']) ->
   {
  "name": "ECCB 2026 toy biomedical knowledge graph",
  "nodes": 6,
  "edges": ...

get_prompt('toy_kg', 'prioritize_genes', ...) ->
   For profile P001, which genes connected to autism spectrum disorder should we pr ...


## 4. Sanity-check one tool (no LLM)

Before involving the model, call a tool straight through the adapter so the
structured evidence is visible. 

In [5]:
phenotype = parse_mcp(await tools_by_name["search_nodes"].ainvoke({"query": "autism"}))
print("search_nodes('autism') ->", phenotype)

candidates = parse_mcp(
    await tools_by_name["rank_candidate_genes"].ainvoke(
        {"profile_id": "P001", "phenotype_id": "HP:0000729"}
    )
)
print("\nrank_candidate_genes(P001, HP:0000729) ->", candidates)

search_nodes('autism') -> [{'node_id': 'HP:0000729', 'name': 'Autism spectrum disorder', 'type': 'phenotype'}]

rank_candidate_genes(P001, HP:0000729) -> [{'gene_id': 'GENE:SHANK3', 'name': 'SHANK3', 'profile_score': 2.4, 'phenotype_id': 'HP:0000729'}, {'gene_id': 'GENE:SCN2A', 'name': 'SCN2A', 'profile_score': 1.7, 'phenotype_id': 'HP:0000729'}, {'gene_id': 'GENE:CHD8', 'name': 'CHD8', 'profile_score': 0.8, 'phenotype_id': 'HP:0000729'}]


## 5. Bind the MCP tools to LLM Agent

Identical to notebook 01 — `bind_tools` doesn't care that these tools came from
an MCP server instead of a `@tool` decorator.

In [ ]:
from langchain_anthropic import ChatAnthropic

# Model backend is not finalised (see planning/session-3-framework-plan.md).
MODEL = "claude-sonnet-4-6"

llm = ChatAnthropic(model=MODEL, temperature=0)
llm_with_tools = llm.bind_tools(tools)
print("Using model:", MODEL)

Using model: claude-sonnet-4-6


## 6. Run the agent loop (async)

Same loop as notebook 01, but `await`-ing the model and the tools because MCP
tools are asynchronous.

In [7]:
import json

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage


async def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> AIMessage:
    """Minimal async tool-calling loop over the MCP-backed Claude model."""
    messages = [
        SystemMessage(
            content=(
                "You are a biomedical research assistant. Use the available "
                "tools to query the knowledge graph and profile data. Always "
                "ground factual claims in tool results, and briefly cite which "
                "tool evidence supports your answer."
            )
        ),
        HumanMessage(content=question),
    ]

    for step in range(max_steps):
        ai_msg = await llm_with_tools.ainvoke(messages)
        messages.append(ai_msg)

        if not ai_msg.tool_calls:
            return ai_msg

        for call in ai_msg.tool_calls:
            if verbose:
                print(f"[step {step}] -> {call['name']}({call['args']})")
            result = await tools_by_name[call["name"]].ainvoke(call["args"])
            messages.append(
                ToolMessage(
                    content=json.dumps(result, default=str),
                    tool_call_id=call["id"],
                )
            )

    raise RuntimeError(f"Agent did not finish within {max_steps} steps.")

In [8]:
question = (
    "For profile P001, which genes connected to autism spectrum disorder "
    "should we prioritize, and why?"
)

final = await run_agent(question)
print("\n=== FINAL ANSWER ===\n")
print(final.content)

[step 0] -> search_nodes({'query': 'autism spectrum disorder'})
[step 0] -> rank_profile_features({'profile_id': 'P001'})
[step 1] -> rank_candidate_genes({'profile_id': 'P001', 'phenotype_id': 'HP:0000729'})
[step 1] -> get_neighbors({'node_id': 'HP:0000729'})

=== FINAL ANSWER ===

Excellent! All the data is in. Here's a comprehensive synthesis:

---

## 🧬 ASD Gene Prioritization for Profile P001

Three genes are connected to **Autism Spectrum Disorder** (`HP:0000729`) in the knowledge graph, all confirmed via literature-backed `associated_with` edges. Here is how they rank for profile P001:

| Rank | Gene | Profile Score | Evidence |
|------|------|--------------|----------|
| 🥇 1 | **SHANK3** | **2.4** | ASD-associated (literature edge) |
| 🥈 2 | **SCN2A** | **1.7** | ASD-associated (literature edge) |
| 🥉 3 | **CHD8** | **0.8** | ASD-associated (literature edge) |

---

### 🔍 Gene-by-Gene Rationale

#### 1. SHANK3 — *Top Priority* (Score: 2.4)
- **Highest profile score** across al

## 7. Same answer as the direct-tools track

The ranking should match notebook 01 (`SHANK3`, `SCN2A`, `CHD8`) — same
backend, different transport.

In [9]:
print("MCP-hosted tools should produce the same ranking as the direct tools:")
for item in candidates:
    print(f"  {item['name']}: profile score {item['profile_score']}")

MCP-hosted tools should produce the same ranking as the direct tools:
  SHANK3: profile score 2.4
  SCN2A: profile score 1.7
  CHD8: profile score 0.8


## Reflection

- **Primitives:** name the three MCP primitives this server exposes. The agent
  loop used `get_tools()` — which adapter calls would load the resource and the
  prompt instead? (We showed it: `get_resources()` and `get_prompt()`.)
- **Did the adapter "throw away" MCP?** No — it *is* an MCP client speaking the
  same protocol as section 2. What's the actual difference between section 2 and
  section 3, then? (Hint: convenience and object model, not capability.)
- **Layering:** MCP rode on top of `stdio` + JSON-RPC here. What would change if
  the transport were HTTP instead? (Hint: the primitives wouldn't.)
- **Annotations:** every tool was tagged `readOnlyHint=True`. How might a real
  client (e.g. Claude Desktop) use that hint? What changes for a tool that
  *writes* to a database?
- **Same answer:** the ranking matched notebook 01 (`SHANK3`, `SCN2A`, `CHD8`).
  So what did MCP actually buy us here — and when would it start to pay off?
  (big data, private/credentialed data, reuse across multiple clients.)
